# Argon A-to-Z

This tutorial demonstrates a-to-z how to optimise Lennard Jones parameters for liquid argon, and without going into details. For details see other tutorials and wider MDMC documentation.

In [ ]:
# Imports used for this tutorial
import numpy as np
import os
from MDMC.control import Control
from MDMC.MD import Atom, Dispersion, LennardJones, Simulation, Universe, Molecule

In [ ]:
# Change the number of threads depending on the number of physical cores on your computer
# as it was tested for LAMMPS
os.environ["OMP_NUM_THREADS"] = "8"

In [ ]:
# Build universe with density 0.0176 atoms per AA^-3
density = 0.0285
# This means cubic universe of side:
# 23.0668 A will contain 216 Ar atoms
# 26.911 A will contain 343 Ar atoms
# 30.7553 A will contain 512 Ar atoms
# 38.4441 A will contain 1000 Ar atoms
universe = Universe(dimensions=33.)
Ar1 = Atom('Ar', charge=0.)
Ar2 = Atom('Ar', charge=0., position=(1.6, 1.6, 1.6))
# Calculating number of Ar atoms needed to obtain density
n_ar_atoms = int(density * np.product(universe.dimensions)/2.)
print(f'Number of argon atoms = {n_ar_atoms}')
Ar_mol = Molecule(atoms=[Ar1, Ar2])
universe.fill(structures=Ar_mol, num_struc_units=(n_ar_atoms))

In the Jupyter cell above, a box of Argon atoms is set up. However, at this point there is no interaction forces between the argon atoms! In the cell below an appropriate (for argon) force-field interaction potential is defined.

In [ ]:
Ar_dispersion = Dispersion(universe,
                           (Ar1.atom_type, Ar1.atom_type), (Ar1.atom_type, Ar2.atom_type), (Ar2.atom_type, Ar2.atom_type),
                           cutoff=8.,
                           function=LennardJones(epsilon=1.0, sigma=3.36))

In this case the interaction potential chosen is the humble Lennard Jones (to get info see doc or type `help(LennardJones)`).

Also, a `cutoff` value is chosen (see `help(Dispersion)` for more info). A [rule of thumb for Lennard-Jones](https://en.wikipedia.org/wiki/Lennard-Jones_potential) is to pick `cutoff=2.5*sigma`. The value for argon is recommended to be between 8 and 12 ang. `cutoff` is not a force-field parameter and therefore will not be refined. Ideally, and for any system you want to pick at value of the `cutoff` which is small while not compromising accuracy. For this system picking a value between 8 and 12 ang is found to give near identifical results.

Next (and before starting the refinement), we set up the MD engine and equilibrate the system. Note with MDMC the equilibration only needs to be done once. 

In [ ]:
# MD Engine setup
simulation = Simulation(universe,
                        engine="lammps",
                        time_step=1.035840042816334,
                        temperature=120.,
                        traj_step=1000)

In [ ]:
# Energy Minimization and equilibration
simulation.minimize(n_steps=5000)
simulation.run(n_steps=5000, equilibration=True)

OK; time to set up the actual refinement of the force-field parameters. 

First we need some data to refine against:

In [ ]:
# exp_datasets is a list of dictionaries with one dictionary per experimental
# dataset
# Dataset from: van Well et al. (1985). Physical Review A, 31(5), 3391-3414
# resolution is None as the original author already accounted for instrument resolution
exp_datasets = [{'file_name':'data/fine_argon_dcsf_step100fs.dat',
                 'type':'SQw',
                 'reader':'MDANSESQw',
                 'weight':1.,
                 'auto_scale':True,
                 'resolution':1.}]

The number of `MD_steps` specified must be large enough to allow for successful calculation of all observables. This depends the `type` of the dataset provided and the value of the `traj_step` (specified when creating the `Simulation`). If a value for `MD_steps` is not provided, then the minimum number needed will be used automatically.

Additionally, some observables will have an upper limit on the number of MD_steps that can be used in calculating their dependent variable(s). In these cases, the number of `MD_steps` is rounded down to a multiple of this upper limit so that we only run steps that will be useful. For example, if we use 1000 `MD_steps` in calculation, but a value of 2500 is provided, then we will run 2000 steps and use this to calculate the variable twice, without wasting time performing an additional 500 steps.

In [ ]:
fit_parameters = universe.parameters
fit_parameters['sigma'].constraints = [2.7,3.8]
fit_parameters['epsilon'].constraints = [0.5, 1.5]


control = Control(simulation=simulation,
                  exp_datasets=exp_datasets,
                  fit_parameters=fit_parameters,
                  minimizer_type="GPO",
                  reset_config=True,
                  MD_steps=40000,
                  equilibration_steps=2500,
                  n_initial = 4)

And finally start the refinement! Bump up `n_steps` from 3 when you are ready.

In [ ]:
# Run the refinement, i.e. refine the FF parameters against the data
control.refine(n_steps=10)
#control.plot_results();

In [ ]:
#control.plot_results();

from skopt.plots import plot_evaluations, plot_objective
_ = plot_evaluations(control.minimizer.optimizer.get_result(), bins=10)
_ = plot_objective(control.minimizer.optimizer.get_result(), n_samples=30, dimensions=['epsilon', 'sigma'])

print(obs_pair_lmp.exp_obs.data)
print(obs_pair_lmp.MD_obs.data)

In [ ]:
rescale_factor

In [ ]:

%matplotlib widget
import matplotlib.pyplot as plt
from matplotlib.widgets import Slider


rescale_factor = control.observable_pairs[0].rescale_factor

SQw=control.observable_pairs[0].exp_obs.data['dependent']['SQw'][0]
SQw_err=control.observable_pairs[0].exp_obs.data['errors']['SQw'][0]
Q=control.observable_pairs[0].exp_obs.data['independent']['Q']
E=control.observable_pairs[0].exp_obs.data['independent']['E']

MD_SQw=control.observable_pairs[0].MD_obs.data['dependent']['SQw'][0]/rescale_factor
MD_SQw_err=control.observable_pairs[0].MD_obs.data['errors']['SQw'][0]/rescale_factor
MD_Q=control.observable_pairs[0].MD_obs.data['independent']['Q']
MD_E=control.observable_pairs[0].MD_obs.data['independent']['E']



fig, ax = plt.subplots()
line = ax.errorbar(E, SQw[0], yerr=SQw_err[0], elinewidth=1, color='black')
line_lmp = ax.errorbar(MD_E, MD_SQw[0], yerr=MD_SQw_err[0], elinewidth=1, color='blue')
ax.set_xlabel('E (meV)')
ax.set_ylabel('S(Q,E) (arb)')
ax.set_title('Argon data')
plt.yscale('log')
fig.subplots_adjust(left=0.25, bottom=0.3)



Q_slider_ax  = fig.add_axes([0.25, 0.15, 0.65, 0.03], facecolor='lightgoldenrodyellow')
Q_slider = Slider(Q_slider_ax, 'Q index', 0, len(Q)-1, valinit=1, valstep=1)
Q_label=plt.text(1,1.7,f'Q={Q[1]} $\AA^{-1}$')
def Q_on_changed(val):
    line.data = (E, SQw[val], E*0.0, SQw_err[val])
    line_lmp.data = (E, MD_SQw[val], E*0.0, MD_SQw_err[val])
    Q_label.set_text(f'Q={Q[val]} $\AA^{-1}$')
    fig.canvas.draw_idle()
    ax.set_ylim(0,max(np.max(SQw[val]),1e-5))
Q_slider.on_changed(Q_on_changed)
plt.show()